# Cold 4-bet strategy analysis

This notebook builds two pre-flop regression models from your 168-scenario dataset:

1. **4-bet range model** using `FOURBET_RANGE_LOGIT`, and
2. **4-bet sizing model** using `FOURBET_SIZE_RELATIVE`.

The workflow covers:

- data loading,
- feature engineering,
- null models,
- forward AIC variable selection,
- fitted-model summaries, and
- actual-versus-expected visual diagnostics by game and positions.

> **Expected input columns**
>
> `RECORD_NO`, `GAME`, `OPEN`, `OPEN_SIZE_ABS`, `3BET`, `3BET_SIZE_ABS`, `4BET`, `4BET_RANGE`, `4BET_SIZE_ABS`


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [ ]:
# --- Configuration ---------------------------------------------------------
# Update DATA_PATH if your attached file lives somewhere else.
DATA_PATH = None

GAME_ORDER = ["CASH 100bb", "MTT ChipEV 80bb", "MTT ICM (50% left) 35bb"]
OPEN_ORDER = ["UTG", "UTG+1", "LJ", "HJ", "CO", "BTN"]
THREEBET_ORDER = ["UTG+1", "LJ", "HJ", "CO", "BTN", "SB"]
FOURBET_ORDER = ["LJ", "HJ", "CO", "BTN", "SB", "BB"]
TABLE_ORDER = ["UTG", "UTG+1", "LJ", "HJ", "CO", "BTN", "SB", "BB"]
POSITION_TO_INDEX = {position: index for index, position in enumerate(TABLE_ORDER)}

EXPECTED_COLUMNS = {
    "RECORD_NO",
    "GAME",
    "OPEN",
    "OPEN_SIZE_ABS",
    "3BET",
    "3BET_SIZE_ABS",
    "4BET",
    "4BET_RANGE",
    "4BET_SIZE_ABS",
}


In [ ]:
def guess_dataset_path() -> Path:
    """Return the most likely dataset path if DATA_PATH is not set manually."""
    search_roots = [Path.cwd(), Path.cwd() / "data", Path.cwd() / "notebooks"]
    patterns = [
        "*4*bet*.csv",
        "*4*bet*.xlsx",
        "*four*bet*.csv",
        "*four*bet*.xlsx",
        "*.csv",
        "*.xlsx",
    ]

    candidates: list[Path] = []
    for root in search_roots:
        if not root.exists():
            continue
        for pattern in patterns:
            candidates.extend(sorted(root.glob(pattern)))

    if not candidates:
        raise FileNotFoundError(
            "Could not find a dataset automatically. Set DATA_PATH to your CSV or Excel file."
        )

    return candidates[0]


def load_fourbet_data(data_path: str | Path | None = None) -> pd.DataFrame:
    """Load the 4-bet dataset, standardise the main column names, and validate schema."""
    path = Path(data_path) if data_path is not None else guess_dataset_path()

    if path.suffix.lower() == ".csv":
        raw = pd.read_csv(path)
    elif path.suffix.lower() in {".xlsx", ".xls"}:
        raw = pd.read_excel(path)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}")

    df = raw.rename(
        columns={
            "3BET": "THREEBET_POS",
            "4BET": "FOURBET_POS",
            "OPEN": "OPEN_POS",
            "3BET_SIZE_ABS": "THREEBET_SIZE_ABS",
            "4BET_RANGE": "FOURBET_RANGE",
        }
    ).copy()

    reverse_lookup = {
        "OPEN_POS": "OPEN",
        "THREEBET_POS": "3BET",
        "FOURBET_POS": "4BET",
    }
    df = df.rename(columns={alias: canonical for alias, canonical in reverse_lookup.items() if alias in df.columns})
    missing = EXPECTED_COLUMNS.difference(df.columns)
    if missing:
        raise ValueError(f"Dataset is missing required columns: {sorted(missing)}")

    df = df.sort_values("RECORD_NO").reset_index(drop=True)
    return df


In [ ]:
def circular_distance(start: str, end: str, order: list[str]) -> int:
    """Measure clockwise distance between two table positions on an 8-max table."""
    start_idx = POSITION_TO_INDEX[start]
    end_idx = POSITION_TO_INDEX[end]
    return (end_idx - start_idx) % len(order)


def prepare_features(df: pd.DataFrame) -> pd.DataFrame:
    """Engineer modelling and visualisation features for the cold 4-bet analysis."""
    data = df.copy()

    # Preserve original names, then create modelling-friendly aliases.
    data = data.rename(columns={"OPEN": "OPEN_POS", "3BET": "THREEBET_POS", "4BET": "FOURBET_POS"})

    # Ordered categorical variables for plotting and modelling.
    data["V_GAME"] = pd.Categorical(data["GAME"], categories=GAME_ORDER, ordered=True)
    data["V_OPEN"] = pd.Categorical(data["OPEN_POS"], categories=OPEN_ORDER, ordered=True)
    data["V_3BET"] = pd.Categorical(data["THREEBET_POS"], categories=THREEBET_ORDER, ordered=True)
    data["V_4BET"] = pd.Categorical(data["FOURBET_POS"], categories=FOURBET_ORDER, ordered=True)

    # Explicit one-hot encodings requested for interpretation / export.
    data = pd.get_dummies(
        data,
        columns=["OPEN_POS", "THREEBET_POS", "FOURBET_POS"],
        prefix=["open", "threebet", "fourbet"],
        dtype=int,
    )

    # Restore readable categorical fields after get_dummies for formula-based modelling.
    data["OPEN_POS"] = data[[f"open_{p}" for p in OPEN_ORDER]].idxmax(axis=1).str.replace("open_", "", regex=False)
    data["THREEBET_POS"] = data[[f"threebet_{p}" for p in THREEBET_ORDER]].idxmax(axis=1).str.replace("threebet_", "", regex=False)
    data["FOURBET_POS"] = data[[f"fourbet_{p}" for p in FOURBET_ORDER]].idxmax(axis=1).str.replace("fourbet_", "", regex=False)

    # Ordered categorical dtype for the restored fields.
    data["OPEN_POS"] = pd.Categorical(data["OPEN_POS"], categories=OPEN_ORDER, ordered=True)
    data["THREEBET_POS"] = pd.Categorical(data["THREEBET_POS"], categories=THREEBET_ORDER, ordered=True)
    data["FOURBET_POS"] = pd.Categorical(data["FOURBET_POS"], categories=FOURBET_ORDER, ordered=True)

    # Response variables.
    eps = 1e-6
    clipped_range = data["FOURBET_RANGE"].clip(eps, 1 - eps)
    data["FOURBET_RANGE_LOGIT"] = np.log(clipped_range / (1 - clipped_range))
    data["FOURBET_SIZE_RELATIVE"] = data["4BET_SIZE_ABS"] / data["3BET_SIZE_ABS"]

    # Size and geometry features.
    data["THREEBET_OVER_OPEN"] = data["3BET_SIZE_ABS"] / data["OPEN_SIZE_ABS"]
    data["FOURBET_OVER_OPEN"] = data["4BET_SIZE_ABS"] / data["OPEN_SIZE_ABS"]
    data["SIZE_GAP_ABS"] = data["4BET_SIZE_ABS"] - data["3BET_SIZE_ABS"]

    data["OPEN_IS_EARLY"] = data["OPEN_POS"].isin(["UTG", "UTG+1", "LJ"]).astype(int)
    data["OPEN_IS_LATE"] = data["OPEN_POS"].isin(["HJ", "CO", "BTN"]).astype(int)
    data["THREEBETTER_IN_BLINDS"] = data["THREEBET_POS"].isin(["SB", "BB"]).astype(int)
    data["FOURBETTER_IN_BLINDS"] = data["FOURBET_POS"].isin(["SB", "BB"]).astype(int)
    data["FOURBETTER_IS_BB"] = (data["FOURBET_POS"] == "BB").astype(int)
    data["FOURBETTER_IS_SB"] = (data["FOURBET_POS"] == "SB").astype(int)
    data["BTN_INVOLVED"] = (
        (data["OPEN_POS"] == "BTN") | (data["THREEBET_POS"] == "BTN") | (data["FOURBET_POS"] == "BTN")
    ).astype(int)

    # Positional spacing features using table order.
    data["OPEN_TO_THREEBET_DISTANCE"] = [
        circular_distance(open_pos, threebet_pos, TABLE_ORDER)
        for open_pos, threebet_pos in zip(data["OPEN_POS"], data["THREEBET_POS"])
    ]
    data["THREEBET_TO_FOURBET_DISTANCE"] = [
        circular_distance(threebet_pos, fourbet_pos, TABLE_ORDER)
        for threebet_pos, fourbet_pos in zip(data["THREEBET_POS"], data["FOURBET_POS"])
    ]
    data["OPEN_TO_FOURBET_DISTANCE"] = [
        circular_distance(open_pos, fourbet_pos, TABLE_ORDER)
        for open_pos, fourbet_pos in zip(data["OPEN_POS"], data["FOURBET_POS"])
    ]

    # Matchup labels can be useful for grouped diagnostics.
    data["OPEN_THREEBET_MATCHUP"] = data["OPEN_POS"].astype(str) + " -> " + data["THREEBET_POS"].astype(str)
    data["THREEBET_FOURBET_MATCHUP"] = data["THREEBET_POS"].astype(str) + " -> " + data["FOURBET_POS"].astype(str)

    return data


In [ ]:
raw_df = load_fourbet_data(DATA_PATH)
analysis_df = prepare_features(raw_df)

print(f"Loaded {len(analysis_df)} rows and {analysis_df.shape[1]} columns.")
display(analysis_df.head())


In [ ]:
summary_table = pd.DataFrame(
    {
        "mean": analysis_df[["FOURBET_RANGE", "FOURBET_RANGE_LOGIT", "FOURBET_SIZE_RELATIVE"]].mean(),
        "std": analysis_df[["FOURBET_RANGE", "FOURBET_RANGE_LOGIT", "FOURBET_SIZE_RELATIVE"]].std(),
        "min": analysis_df[["FOURBET_RANGE", "FOURBET_RANGE_LOGIT", "FOURBET_SIZE_RELATIVE"]].min(),
        "max": analysis_df[["FOURBET_RANGE", "FOURBET_RANGE_LOGIT", "FOURBET_SIZE_RELATIVE"]].max(),
    }
)
display(summary_table.round(4))


## Model specification

We estimate two OLS models:

- `FOURBET_RANGE_LOGIT ~ features`
- `FOURBET_SIZE_RELATIVE ~ features`

Each model starts from a null/intercept-only specification and then uses **forward AIC selection** over a set of poker-motivated candidate terms.


In [ ]:
@dataclass
class SelectionResult:
    response: str
    null_formula: str
    selected_formula: str
    history: pd.DataFrame
    model: object


def fit_ols(formula: str, data: pd.DataFrame):
    """Fit an OLS model via statsmodels and return the fitted result."""
    return smf.ols(formula=formula, data=data).fit()


def forward_aic_selection(response: str, candidate_terms: Iterable[str], data: pd.DataFrame) -> SelectionResult:
    """Greedy forward selection using AIC as the stopping criterion."""
    null_formula = f"{response} ~ 1"
    current_terms: list[str] = []
    best_model = fit_ols(null_formula, data)
    best_aic = best_model.aic
    history: list[dict[str, object]] = [
        {"step": 0, "action": "start", "term": "Intercept only", "aic": best_aic}
    ]

    remaining = list(candidate_terms)
    step = 0

    while remaining:
        scored_candidates: list[tuple[float, str, object]] = []
        for term in remaining:
            candidate_formula = f"{response} ~ {' + '.join(current_terms + [term])}"
            candidate_model = fit_ols(candidate_formula, data)
            scored_candidates.append((candidate_model.aic, term, candidate_model))

        scored_candidates.sort(key=lambda item: item[0])
        candidate_aic, candidate_term, candidate_model = scored_candidates[0]

        if candidate_aic + 1e-9 < best_aic:
            step += 1
            current_terms.append(candidate_term)
            remaining.remove(candidate_term)
            best_aic = candidate_aic
            best_model = candidate_model
            history.append({"step": step, "action": "add", "term": candidate_term, "aic": best_aic})
        else:
            break

    selected_formula = f"{response} ~ {' + '.join(current_terms)}" if current_terms else null_formula
    return SelectionResult(
        response=response,
        null_formula=null_formula,
        selected_formula=selected_formula,
        history=pd.DataFrame(history),
        model=best_model,
    )


In [ ]:
COMMON_TERMS = [
    "C(V_GAME)",
    "C(V_OPEN)",
    "C(V_3BET)",
    "C(V_4BET)",
    "OPEN_SIZE_ABS",
    "Q('3BET_SIZE_ABS')",
    "THREEBET_OVER_OPEN",
    "FOURBET_OVER_OPEN",
    "SIZE_GAP_ABS",
    "OPEN_IS_EARLY",
    "OPEN_IS_LATE",
    "THREEBETTER_IN_BLINDS",
    "FOURBETTER_IN_BLINDS",
    "FOURBETTER_IS_SB",
    "FOURBETTER_IS_BB",
    "BTN_INVOLVED",
    "OPEN_TO_THREEBET_DISTANCE",
    "THREEBET_TO_FOURBET_DISTANCE",
    "OPEN_TO_FOURBET_DISTANCE",
    "C(V_GAME):THREEBET_OVER_OPEN",
    "C(V_GAME):OPEN_TO_FOURBET_DISTANCE",
    "C(V_4BET):THREEBETTER_IN_BLINDS",
    "C(V_OPEN):C(V_3BET)",
    "C(V_3BET):C(V_4BET)",
]


In [ ]:
range_result = forward_aic_selection(
    response="FOURBET_RANGE_LOGIT",
    candidate_terms=COMMON_TERMS,
    data=analysis_df,
)

size_result = forward_aic_selection(
    response="FOURBET_SIZE_RELATIVE",
    candidate_terms=COMMON_TERMS,
    data=analysis_df,
)

print("Range model formula:")
print(range_result.selected_formula)
print("\nSize model formula:")
print(size_result.selected_formula)


In [ ]:
null_range_model = fit_ols(range_result.null_formula, analysis_df)
null_size_model = fit_ols(size_result.null_formula, analysis_df)

comparison_table = pd.DataFrame(
    [
        {
            "model": "Range null",
            "formula": range_result.null_formula,
            "aic": null_range_model.aic,
            "r_squared": null_range_model.rsquared,
            "adj_r_squared": null_range_model.rsquared_adj,
        },
        {
            "model": "Range selected",
            "formula": range_result.selected_formula,
            "aic": range_result.model.aic,
            "r_squared": range_result.model.rsquared,
            "adj_r_squared": range_result.model.rsquared_adj,
        },
        {
            "model": "Size null",
            "formula": size_result.null_formula,
            "aic": null_size_model.aic,
            "r_squared": null_size_model.rsquared,
            "adj_r_squared": null_size_model.rsquared_adj,
        },
        {
            "model": "Size selected",
            "formula": size_result.selected_formula,
            "aic": size_result.model.aic,
            "r_squared": size_result.model.rsquared,
            "adj_r_squared": size_result.model.rsquared_adj,
        },
    ]
)
display(comparison_table.round(4))


In [ ]:
print("Forward-selection history: range model")
display(range_result.history)
print("\nForward-selection history: size model")
display(size_result.history)


In [ ]:
print(range_result.model.summary())


In [ ]:
print(size_result.model.summary())


In [ ]:
analysis_df["range_pred_logit"] = range_result.model.predict(analysis_df)
analysis_df["range_pred"] = 1 / (1 + np.exp(-analysis_df["range_pred_logit"]))
analysis_df["size_pred_relative"] = size_result.model.predict(analysis_df)

analysis_df[[
    "RECORD_NO",
    "FOURBET_RANGE",
    "range_pred",
    "FOURBET_SIZE_RELATIVE",
    "size_pred_relative",
]].head()


## Actual versus expected diagnostics

The plots below compare the mean observed outcome to the mean fitted value across the main visualisation variables:

- `V_GAME`
- `V_OPEN`
- `V_3BET`
- `V_4BET`

If the model captures the broad structural trends well, the two lines should track each other closely by category.


In [ ]:
def grouped_actual_vs_expected(
    data: pd.DataFrame,
    group_col: str,
    actual_col: str,
    predicted_col: str,
    title: str,
    ax: plt.Axes,
) -> None:
    """Plot grouped observed vs fitted means for one categorical variable."""
    grouped = (
        data.groupby(group_col, observed=False)
        .agg(actual=(actual_col, "mean"), predicted=(predicted_col, "mean"), n=(actual_col, "size"))
        .reset_index()
    )

    sns.lineplot(data=grouped, x=group_col, y="actual", marker="o", linewidth=2.5, label="Actual", ax=ax)
    sns.lineplot(data=grouped, x=group_col, y="predicted", marker="o", linewidth=2.5, label="Expected", ax=ax)
    ax.set_title(title)
    ax.set_xlabel(group_col)
    ax.set_ylabel(actual_col)
    ax.tick_params(axis="x", rotation=30)


def plot_diagnostics(data: pd.DataFrame, actual_col: str, predicted_col: str, prefix: str) -> None:
    """Create four actual-vs-expected plots across the main visualisation dimensions."""
    fig, axes = plt.subplots(2, 2, figsize=(18, 12), constrained_layout=True)
    grouped_actual_vs_expected(data, "V_GAME", actual_col, predicted_col, f"{prefix}: by game", axes[0, 0])
    grouped_actual_vs_expected(data, "V_OPEN", actual_col, predicted_col, f"{prefix}: by opener", axes[0, 1])
    grouped_actual_vs_expected(data, "V_3BET", actual_col, predicted_col, f"{prefix}: by 3-bettor", axes[1, 0])
    grouped_actual_vs_expected(data, "V_4BET", actual_col, predicted_col, f"{prefix}: by 4-bettor", axes[1, 1])
    handles, labels = axes[0, 0].get_legend_handles_labels()
    for ax in axes.ravel():
        legend = ax.get_legend()
        if legend is not None:
            legend.remove()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=True)
    plt.show()


In [ ]:
plot_diagnostics(
    data=analysis_df,
    actual_col="FOURBET_RANGE",
    predicted_col="range_pred",
    prefix="4-bet range model",
)


In [ ]:
plot_diagnostics(
    data=analysis_df,
    actual_col="FOURBET_SIZE_RELATIVE",
    predicted_col="size_pred_relative",
    prefix="4-bet sizing model",
)


In [ ]:
# Optional extra diagnostics: residual plots for both final models.
fig, axes = plt.subplots(1, 2, figsize=(16, 6), constrained_layout=True)

sns.scatterplot(x=analysis_df["range_pred"], y=range_result.model.resid, ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Range model residuals vs fitted")
axes[0].set_xlabel("Fitted 4-bet range")
axes[0].set_ylabel("Residual (logit scale)")

sns.scatterplot(x=analysis_df["size_pred_relative"], y=size_result.model.resid, ax=axes[1])
axes[1].axhline(0, color="black", linestyle="--", linewidth=1)
axes[1].set_title("Sizing model residuals vs fitted")
axes[1].set_xlabel("Fitted relative 4-bet size")
axes[1].set_ylabel("Residual")

plt.show()


## Interpretation notes

When you run the notebook on your attached dataset, focus on:

- the null-model intercept for the baseline average 4-bet frequency,
- which terms survive the AIC selection process,
- the sign and magnitude of the selected coefficients,
- whether game type shifts both frequency and sizing,
- whether blind cold 4-bets behave differently from in-position cold 4-bets, and
- whether the actual-versus-expected lines stay aligned across opener / 3-bettor / 4-bettor positions.

If you want a stricter or more explainable model, a good next extension is to compare this forward-AIC approach with:

1. a manually curated small model,
2. Lasso / elastic net on the engineered design matrix, and
3. cross-validated out-of-sample error by leave-one-scenario-out or grouped folds.
